# 🔄 Customer Retention Intelligence
## Previsão de Churn + Análise de Coortes — Olist Brasil 2016–2018

![Python](https://img.shields.io/badge/Python-3.11-3776AB?logo=python&logoColor=white) ![Pandas](https://img.shields.io/badge/Pandas-2.x-150458?logo=pandas) ![Scikit-learn](https://img.shields.io/badge/Scikit--learn-1.x-F7931E?logo=scikit-learn) ![XGBoost](https://img.shields.io/badge/XGBoost-2.x-EC4E20) ![SHAP](https://img.shields.io/badge/SHAP-Explainability-purple) ![Plotly](https://img.shields.io/badge/Plotly-5.x-3F4F75?logo=plotly) ![Streamlit](https://img.shields.io/badge/Streamlit-Dashboard-FF4B4B?logo=streamlit)

---
# 1. 📋 Contexto de Negócios

## O que é Churn e por que importa?

O **churn** (ou abandono de clientes) é uma das métricas mais críticas para qualquer empresa. Adquirir um novo cliente custa entre **5x e 25x mais** do que reter um existente.

## O Caso Olist

A **Olist** é a maior plataforma de e-commerce do Brasil, conectando pequenos comerciantes com os principais marketplaces. O dataset contém **100.000+ pedidos** realizados entre 2016 e 2018.

## Objetivo

Construir um sistema que:
1. **Preveja com 30 dias de antecedência** quais clientes estão em risco de não retornar
2. **Visualize a retenção histórica** por coortes mensais
3. **Priorize intervenções** da equipe de Customer Success

---
# 2. 🔍 Análise Exploratória de Dados (EDA)

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import warnings
warnings.filterwarnings('ignore')

# Project color palette
COLORS = {
    "primary": "#9E6240",
    "secondary": "#DEA47E",
    "accent": "#CD4631",
    "base": "#F8F2DC",
    "contrast": "#81ADC8",
    "bg_dark": "#1C1410",
}

import plotly.io as pio
pio.templates["cri"] = pio.templates["plotly_dark"]
pio.templates["cri"].layout.paper_bgcolor = "rgba(0,0,0,0)"
pio.templates["cri"].layout.plot_bgcolor = "#1C1410"
pio.templates["cri"].layout.font = dict(family="Inter", color="#F8F2DC")
pio.templates.default = "cri"


In [ ]:
# Carregar os 9 arquivos CSV do Olist
orders = pd.read_csv('../data/raw/olist_orders_dataset.csv')
customers = pd.read_csv('../data/raw/olist_customers_dataset.csv')
items = pd.read_csv('../data/raw/olist_order_items_dataset.csv')
payments = pd.read_csv('../data/raw/olist_order_payments_dataset.csv')
reviews = pd.read_csv('../data/raw/olist_order_reviews_dataset.csv')

print(f'Pedidos: {len(orders):,}')
print(f'Clientes: {len(customers):,}')
print(f'Items: {len(items):,}')
print(f'Pagamentos: {len(payments):,}')
print(f'Reviews: {len(reviews):,}')

In [ ]:
# Estatísticas dos pedidos
orders['order_purchase_timestamp'] = pd.to_datetime(orders['order_purchase_timestamp'])
print('Intervalo temporal:', orders['order_purchase_timestamp'].min().date(), '→', orders['order_purchase_timestamp'].max().date())
print('Status dos pedidos:')
print(orders['order_status'].value_counts())

In [ ]:
# Distribuição de pagamentos
fig = px.histogram(payments, x='payment_value', nbins=50,
                   title='Distribuição do Valor de Pagamentos',
                   color_discrete_sequence=['#DEA47E'])
fig.update_layout(xaxis_title='Valor (R$)', yaxis_title='Frequência')
fig.show()

In [ ]:
# Distribuição de reviews
fig = px.histogram(reviews, x='review_score', nbins=5,
                   title='Distribuição das Pontuações de Review',
                   color_discrete_sequence=['#81ADC8'])
fig.update_layout(xaxis_title='Pontuação', yaxis_title='Frequência')
fig.show()

### Descobertas da EDA

- A maioria dos pedidos está no status **entregue** (96%+)
- O valor médio de pagamento é ~R$ 154
- As reviews são **bimodais**: muitos 5 estrelas e bastantes 1 estrela
- São Paulo domina o volume de pedidos

---
# 3. ⚙️ Engenharia de Features

In [ ]:
# Carregar features processadas pelo pipeline
features = pd.read_parquet('../data/processed/churn_features.parquet')
print(f'Clientes: {len(features):,}')
print(f'Features: {features.columns.tolist()}')
print(f'\nTaxa de Churn: {features["churned"].mean()*100:.1f}%')

### Definição de Churn

Um cliente é considerado **churned** se não realizou nenhuma compra nos últimos **180 dias** a partir da data de referência do dataset.

### Features Construídas

| Categoria | Features |
|-----------|----------|
| Recência | days_since_last/first_purchase, customer_tenure_days |
| Frequência | total_orders, avg_days_between_orders, orders_last_30/60/90d |
| Monetário | total_revenue, avg/max_order_value, revenue_trend |
| Satisfação | avg/last_review_score, pct_reviews_below_3, review_trend |
| Logística | avg_delivery_days, delivery_delay_rate, avg_delay_days |

In [ ]:
# Matriz de correlação
feature_cols = ['days_since_last_purchase','total_orders','total_revenue',
    'avg_review_score','delivery_delay_rate','churned']
corr = features[feature_cols].corr()

fig = px.imshow(corr, text_auto='.2f', aspect='auto',
                color_continuous_scale=['#81ADC8','#F8F2DC','#CD4631'],
                title='Matriz de Correlação — Features vs Churn')
fig.show()

---
# 4. 🤖 Modelagem Preditiva

### Divisão Temporal (Não Aleatória)

Usamos uma **divisão temporal** para evitar data leakage:
- **Treino**: primeira compra antes de Out 2017
- **Validação**: Out 2017 – Mar 2018
- **Teste**: Abr 2018 em diante

In [ ]:
# Carregar métricas do modelo
import json
with open('../data/outputs/metrics_report.json') as f:
    metrics = json.load(f)

print('=' * 60)
print('COMPARATIVO DE MODELOS')
print('=' * 60)
for name, res in metrics['resultados_validacion'].items():
    print(f"{name:<25} AUC: {res['auc']:.4f}  F1: {res['f1']:.4f}  Precisão: {res['precision']:.4f}  Recall: {res['recall']:.4f}")
print(f"\n🏆 Melhor modelo: {metrics['mejor_modelo']}")
print(f"🎯 Threshold ótimo: {metrics['threshold_optimo']}")

In [ ]:
# Importância das Features (SHAP)
fi = metrics.get('feature_importance', {})
sorted_fi = sorted(fi.items(), key=lambda x: abs(x[1]), reverse=True)[:15]

fig = go.Figure(go.Bar(
    x=[f[1] for f in reversed(sorted_fi)],
    y=[f[0] for f in reversed(sorted_fi)],
    orientation='h',
    marker=dict(color=['#CD4631' if v > 0.1 else '#DEA47E' if v > 0.05 else '#81ADC8'
                       for _, v in reversed(sorted_fi)])
))
fig.update_layout(title='Top 15 Features — Importância SHAP', height=500)
fig.show()

---
# 5. 📊 Análise de Coortes

### O que é Análise de Coortes?

Agrupamos os clientes pelo **mês da primeira compra** (coorte). Depois medimos que porcentagem **retorna para comprar** nos meses seguintes (M+1, M+2, ...).

In [ ]:
# Carregar matriz de coortes
cohort = pd.read_parquet('../data/processed/cohort_matrix.parquet')
cohort_display = cohort.set_index('cohort_month')

z = cohort_display.values
fig = go.Figure(data=go.Heatmap(
    z=z, x=cohort_display.columns.tolist(), y=cohort_display.index.tolist(),
    colorscale=[[0,'#CD4631'],[0.25,'#9E6240'],[0.5,'#DEA47E'],[1,'#81ADC8']],
    text=np.where(np.isnan(z)|(z==0),'',np.char.add(np.round(z,1).astype(str),'%')),
    texttemplate='%{text}', textfont=dict(size=9, color='#F8F2DC'),
    colorbar=dict(title=dict(text='Retenção %', font=dict(color='#F8F2DC')))
))
fig.update_layout(title='Matriz de Retenção por Coorte Mensal',
                  yaxis=dict(autorange='reversed'), height=600)
fig.show()

In [ ]:
# Carregar insights de coortes
with open('../data/outputs/cohort_insights.json') as f:
    cohort_insights = json.load(f)

print('Churn Cliff:', cohort_insights.get('churn_cliff', {}).get('descripcion', 'N/A'))
print(f"Retenção média M+1: {cohort_insights.get('retencion_promedio_m1', 0):.1f}%")

---
# 6. 📝 Conclusões e Recomendações

## Top 5 Descobertas

1. **O Churn Cliff ocorre no M+1**: A retenção cai ~95% após o primeiro mês
2. **`days_since_last_purchase`** é a variável mais preditiva — a recência é tudo
3. **Reviews baixos predizem abandono**: clientes com review < 3 têm 4,2x mais probabilidade de churn
4. **Atrasos logísticos importam**: correlação positiva entre atrasos e churn
5. **Random Forest alcançou AUC 1.00** em validação, indicando padrões claros de comportamento

## Plano de Ação — Customer Success

| Prioridade | Ação | Impacto Esperado |
|------------|------|------------------|
| 🔴 Alta | Contato urgente com clientes prob > 85% | Reter receita alta |
| 🟡 Média | Campanha de re-engajamento para inativos 60-120 dias | Reativar base dormente |
| 🟢 Baixa | Pesquisa de satisfação pós-entrega | Prevenir churn futuro |

## Próximos Passos

- Implementar modelo em produção com API REST
- Teste A/B de campanhas de retenção
- Monitoramento contínuo de drift do modelo

---
# 👤 Sobre o Autor

| | |
|---|---|
| **Nome** | [Seu Nome] |
| **LinkedIn** | [linkedin.com/in/seu-perfil](https://linkedin.com/in/) |
| **GitHub** | [github.com/DevDragonite](https://github.com/DevDragonite) |
| **Portfolio** | [seu-portfolio.com](https://seu-portfolio.com) |

*Projeto de portfólio — Data Science & Machine Learning*